# 파티 코치 LoRA 파인튜닝 — Colab 무료 · T4 16GB

Qwen3.5-4B + Unsloth. **4bit QLoRA (fp16, T4는 bf16 미지원), 최대 길이 8192토큰**.
같은 내용의 다른 버전: `finetune_laptop`(RTX 4050) · `finetune_home`(RTX 3060) · `finetune_colab`(T4). 설정 셀만 다르다.

`llm/agent`의 **시스템 프롬프트·도구 정의를 그대로** 써서, 모델이 도구 호출 방식과 파티 판단을 배우게 한다.
사실(종족값·사용률 등)은 외우게 하지 않는다. 학습 데이터 속 도구 결과는 항상 실제 back API 값이다.

> Google Drive에 묶음 zip을 두고 체크포인트·어댑터도 드라이브에 저장한다. 세션이 끊겨도 1번부터 다시 실행하면 7번에서 이어서 학습.

**준비**
- Windows 저장소에서 묶음 만들기 (back 코드·DB·코치 도구·학습 데이터를 zip 하나로):
  ```
  cd llm
  ..\.venv\Scripts\python training\make_data_v2.py --export 220   # → llm/training/finetuning_data_ver2/
  ..\.venv\Scripts\python training\pack_colab.py                  # → llm/training/colab_bundle.zip
  ```
- Google Drive의 `내 드라이브/pokemon_coach/` 폴더에 `colab_bundle.zip`을 올려 둔다 (한 번만. 데이터를 다시 만들면 덮어쓰기)
- Colab: 파일 → 노트북 업로드로 이 노트북 열기 → 런타임 → 런타임 유형 변경 → **T4 GPU** → 위에서부터 실행
  (1번 셀에서 드라이브 연결 허용 창이 뜬다)
- 체크포인트·어댑터도 드라이브(`pokemon_coach/outputs`)에 저장되니, 세션이 끊겨도 7번 셀을 다시 돌리면 이어서 학습한다
- 드라이브를 안 쓰려면 1번 셀의 `USE_DRIVE = False`, 왼쪽 파일 탭에 zip을 올린다 (끊기면 처음부터)

**순서**: 1 설치 → 2 설정(학습률 등) → 3~4 도구·데이터 → 5 모델 → 6 길이 확인 → 7 학습(실시간 loss·학습률) → 7-1 그래프
→ 8 평가 → 9 저장 → **10 채팅방**(학습한 모델과 실제 도구로 대화)

## 1. 드라이브 연결 + 설치 + 묶음 풀기
런타임마다 새로 설치한다 (몇 분 걸림).

In [ ]:
import os, sys, json, re, shutil, subprocess
from pathlib import Path

USE_DRIVE = True                                     # 드라이브에 zip을 두고, 체크포인트·어댑터도 드라이브에 저장
DRIVE_DIR = Path('/content/drive/MyDrive/pokemon_coach')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth', 'gradio',
                'django>=5.2,<6', 'djangorestframework'], check=True)

# 묶음은 런타임 디스크에 풀어서 쓴다 (DB를 드라이브에서 바로 읽으면 느림). 드라이브·파일 탭 중 최신 zip
zips = sorted([*Path('/content').glob('colab_bundle*.zip'), *(DRIVE_DIR.glob('colab_bundle*.zip') if USE_DRIVE else [])],
              key=lambda p: p.stat().st_mtime)
if not zips:
    raise FileNotFoundError(f'colab_bundle.zip이 없음 — {DRIVE_DIR if USE_DRIVE else "왼쪽 파일 탭"}에 올릴 것 '
                            '(llm/training/pack_colab.py로 만듦)')
REPO = Path('/content/repo')
shutil.rmtree(REPO, ignore_errors=True)
shutil.unpack_archive(zips[-1], REPO)
print('묶음:', zips[-1], '→', sorted(p.name for p in REPO.iterdir()))

## 2. 설정
학습을 바꾸고 싶으면 여기 값만 고친다. 셀을 실행하면 표로 다시 보여 준다.
- **학습률(LR)**: 한 번에 가중치를 얼마나 크게 고칠지. LoRA는 보통 1e-4~2e-4. loss가 튀거나 이상한 말을 하면 낮추고, 너무 안 내려가면 올린다
- **학습량**: 시간 ≈ 대화 수 × 에폭. T4는 대화 하나에 약 25초 (900개 × 2에폭 ≈ 12시간이라 300개 × 1에폭으로 시작)
- **에폭(EPOCHS)**: 데이터를 몇 바퀴 볼지. 많으면 데이터를 외워서 평가 loss가 다시 올라간다 (7-1 그래프에서 확인)
- **LoRA r**: 붙이는 학습 파라미터 크기. 클수록 많이 배우지만 메모리·과적합 위험이 커진다

In [ ]:
import torch

MODEL_NAME = 'unsloth/Qwen3.5-4B'          # 원본: Qwen/Qwen3.5-4B (Apache 2.0, 이미지도 받는 모델이라 FastModel로 불러옴)
LOAD_IN_4BIT = True
MAX_SEQ_LEN = 8192                     # 넘는 대화는 학습에서 뺀다 (6번 셀). 시스템 프롬프트+도구 설명만 약 2,400토큰

LR = 2e-4                                  # 학습률
LR_SCHEDULER = 'cosine'                    # 학습률을 처음 WARMUP 동안 올렸다가 코사인으로 0까지 줄임
WARMUP_RATIO = 0.05
EPOCHS = 1                                 # 답변이 틀에 맞춘 문장이라 금방 배움. 2 이상은 외울 위험
TRAIN_LIMIT = None                          # 학습 턴 수 제한 (None = 전부). T4는 한 턴(약 5천 토큰)에 약 20초
BATCH, GRAD_ACC = 1, 8                     # 실제 배치 = BATCH × GRAD_ACC (메모리 때문에 1개씩 8번 모아서 한 번 업데이트)
LORA_R, LORA_ALPHA = 16, 16
WEIGHT_DECAY = 0.01
LOG_EVERY = 5                              # 몇 스텝마다 loss·학습률을 찍을지
EVAL_N = 30                                # 학습 중 평가 loss를 잴 평가 대화 수 (많으면 느려짐)
SEED = 3407

BF16 = torch.cuda.get_device_capability()[0] >= 8   # bf16은 Ampere(RTX 30xx) 이상. T4는 fp16
                                                    # (is_bf16_supported()는 T4에서도 True라 쓰지 않음)
DATA_DIR = REPO / 'llm/training/finetuning_data_ver2'     # make_data_v2.py --export
OUT_DIR = (DRIVE_DIR if USE_DRIVE else Path('/content')) / 'outputs'          # 체크포인트·어댑터
RESUME = True                              # 체크포인트가 있으면 이어서 학습 (설정·데이터를 바꿨으면 False 또는 checkpoints 폴더 삭제)

gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 2**30
if 'T4' not in gpu.name:
    print(f'⚠ 이 노트북은 T4용인데 지금 GPU는 {gpu.name} ({VRAM_GB:.1f}GB). 맞는 버전을 쓰거나 위 값을 고칠 것')

def show_settings(**extra):
    rows = [('GPU', f'{gpu.name} ({VRAM_GB:.1f}GB)'), ('모델', MODEL_NAME),
            ('방식', '4bit QLoRA' if LOAD_IN_4BIT else 'LoRA (16bit)'), ('정밀도', 'bf16' if BF16 else 'fp16'),
            ('최대 길이', MAX_SEQ_LEN), ('학습률', f'{LR:g} ({LR_SCHEDULER}, warmup {WARMUP_RATIO:.0%})'),
            ('에폭', EPOCHS), ('학습 대화 제한', TRAIN_LIMIT or '없음 (전부)'), ('배치', f'{BATCH} × {GRAD_ACC} = {BATCH * GRAD_ACC}'),
            ('LoRA r / alpha', f'{LORA_R} / {LORA_ALPHA}'), ('weight decay', WEIGHT_DECAY), *extra.items()]
    w = max(len(k) for k, _ in rows)
    print('\n'.join(f'{k:<{w}}  {v}' for k, v in rows))

show_settings()

## 3. 시스템 프롬프트·도구 불러오기
서버(`llm/agent`)와 같은 프롬프트·도구로 학습해야 붙였을 때 그대로 동작한다.
도구 정의는 Claude 형식(`input_schema`)이라 Qwen 형식(`function.parameters`)으로 바꾼다.

In [ ]:
for p in (REPO / 'llm', REPO / 'llm/training'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from agent.prompt import SYSTEM, screen_context
from agent.tools import TOOLS as CLAUDE_TOOLS

TOOLS = [{'type': 'function', 'function': {'name': t['name'], 'description': t['description'],
                                           'parameters': t['input_schema']}} for t in CLAUDE_TOOLS]
print('도구', len(TOOLS), '개:', ', '.join(t['function']['name'] for t in TOOLS))

## 4. 학습 데이터
`make_data_v2.py --export`가 만든 `finetuning_data_ver2/train.jsonl`(학습)과 `eval.jsonl`(평가).
한 줄 = 대화의 한 턴. 지난 턴은 질문·답변 글만 있고(서버와 같게), 마지막 질문 앞에 [화면] 상황이 붙는다.
학습/평가는 원본 팀(group) 단위로 나뉘어 있고, 사용자가 준 질문 5개는 평가에 들어 있다.
```json
{"id": "v2-0007-t2", "kind": "fill_one", "format": "singles", "group": "team:38561", "turn": 2, "messages": [
  {"role": "user", "content": "이 파티에 부족한 포지션 채워줘"},
  {"role": "assistant", "content": "지금 파티: …"},
  {"role": "user", "content": "[화면] 포맷: 싱글 …\n\n막이 말고"},
  {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": "find_partners", "arguments": {…}}}]},
  {"role": "tool", "content": "<실제 도구 결과 JSON>"},
  …
  {"role": "assistant", "content": "최종 답변"}
]}
```
시스템 프롬프트는 파일에 없고 여기서 넣는다.

In [ ]:
from collections import Counter

def read(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'{path} 없음 — make_data_v2.py --export로 먼저 만들 것')
    rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    for r in rows:
        if r['messages'][0]['role'] != 'system':
            r['messages'].insert(0, {'role': 'system', 'content': SYSTEM})
    return rows

train_rows, eval_rows = read('train.jsonl')[:TRAIN_LIMIT], read('eval.jsonl')
print(f'학습 {len(train_rows)}턴 / 평가 {len(eval_rows)}턴')
print(Counter(r['kind'] for r in train_rows))

## 5. 모델 불러오기 + LoRA
Qwen3.5는 이미지도 받는 모델이라 `FastModel`로 불러오고, 언어 부분에만 LoRA를 붙인다.

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, full_finetuning=False)
tok = getattr(tokenizer, 'tokenizer', tokenizer)     # processor면 안쪽 글자 tokenizer

model = FastModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=SEED)
model.print_trainable_parameters()
print(f'모델 올린 뒤 VRAM {torch.cuda.memory_reserved() / 2**30:.1f}GB')

## 6. 채팅 템플릿 적용 + 길이 확인
Qwen3.5 템플릿이 도구 정의는 시스템 프롬프트에, 도구 호출은 `<tool_call><function=이름><parameter=키>값</parameter>...`,
결과는 user 차례의 `<tool_response>`로 넣는다. thinking은 끈다 (서버에서도 끄고 쓸 것).

In [ ]:
import math
from datasets import Dataset

def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(messages, tools=TOOLS, tokenize=False,
                                   add_generation_prompt=add_generation_prompt, enable_thinking=False)

def to_dataset(rows, label):
    texts = [render(r['messages']) for r in rows]
    lens = [len(tok(t).input_ids) for t in texts]
    keep = [t for t, n in zip(texts, lens) if n <= MAX_SEQ_LEN]
    dropped = Counter(r['kind'] for r, n in zip(rows, lens) if n > MAX_SEQ_LEN)
    print(f'{label}: 토큰 최대 {max(lens)} 평균 {sum(lens) // len(lens)} | {MAX_SEQ_LEN} 넘어서 뺀 것 {len(texts) - len(keep)}개 {dict(dropped)}')
    return Dataset.from_dict({'text': keep})

dataset = to_dataset(train_rows, '학습')
eval_dataset = to_dataset(eval_rows[:EVAL_N], '평가')

STEPS = math.ceil(len(dataset) / (BATCH * GRAD_ACC)) * EPOCHS
WARMUP_STEPS = math.ceil(STEPS * WARMUP_RATIO)
EVAL_EVERY = max(STEPS // 5, 1)          # 학습 중 평가 loss를 5번쯤 잰다
show_settings(**{'학습 대화': len(dataset), '평가 대화': len(eval_dataset), '총 스텝': STEPS,
                 'warmup 스텝': WARMUP_STEPS, '평가 loss 주기': f'{EVAL_EVERY}스텝마다'})
print('\n예시 (끝부분):\n' + dataset[0]['text'][-800:])

## 7. 학습
답변(assistant) 부분만 loss를 계산한다. 사용자 질문·도구 결과(`<tool_response>`, user 차례로 들어감)는 제외.
`LOG_EVERY` 스텝마다 **loss · 학습률 · grad norm · VRAM · 남은 시간**을 한 줄씩 찍는다. 평가 loss는 `EVAL_EVERY`마다.
- loss가 꾸준히 내려가면 정상. 평가 loss가 올라가기 시작하면 외우는 중 (에폭·학습률을 줄인다)
- grad norm이 갑자기 커지면 학습률이 높은 것

In [ ]:
import time
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

class LiveLog(TrainerCallback):
    """로그가 나올 때마다 한 줄로: 스텝, 에폭, loss, 평가 loss, 학습률, grad norm, VRAM, 남은 시간."""
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = self.step_end = time.time()
        self.start_step, self.eval_time, self.evals = state.global_step, 0.0, 0

    def on_step_end(self, args, state, control, **kw):
        self.step_end = time.time()

    def on_evaluate(self, args, state, control, **kw):       # 평가 시간은 따로 (남은 시간이 튀지 않게)
        self.eval_time += time.time() - self.step_end
        self.evals += 1

    def on_log(self, args, state, control, logs=None, **kw):
        if not logs or 'train_runtime' in logs:
            return
        s = state.global_step
        done = max(s - self.start_step, 1)
        per_step = (time.time() - self.t0 - self.eval_time) / done
        evals_left = (state.max_steps - s) // max(args.eval_steps or state.max_steps, 1)
        left = per_step * (state.max_steps - s) + evals_left * self.eval_time / max(self.evals, 1)
        parts = [f'step {s:>4}/{state.max_steps}', f'epoch {state.epoch:.2f}']
        if 'loss' in logs: parts.append(f'loss {logs["loss"]:.4f}')
        if 'eval_loss' in logs: parts.append(f'★ 평가 loss {logs["eval_loss"]:.4f}')
        if 'learning_rate' in logs: parts.append(f'lr {logs["learning_rate"]:.2e}')
        if 'grad_norm' in logs: parts.append(f'grad {logs["grad_norm"]:.2f}')
        parts += [f'VRAM {torch.cuda.max_memory_reserved() / 2**30:.1f}GB', f'남은 {left / 60:.0f}분']
        print(' | '.join(parts), flush=True)

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=dataset, eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field='text', output_dir=str(OUT_DIR / 'checkpoints'),
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=1, gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type=LR_SCHEDULER, warmup_steps=WARMUP_STEPS,
        optim='adamw_8bit', weight_decay=WEIGHT_DECAY, bf16=BF16, fp16=not BF16,
        logging_steps=LOG_EVERY, eval_strategy='steps', eval_steps=EVAL_EVERY,
        save_strategy='steps', save_steps=EVAL_EVERY, save_total_limit=2,      # 끊겨도 이어서 학습할 수 있게 자주 저장
        seed=SEED, report_to='none', disable_tqdm=True),
    callbacks=[LiveLog()],
)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')
from transformers.trainer_utils import get_last_checkpoint
ckpt = get_last_checkpoint(str(OUT_DIR / 'checkpoints')) if RESUME and (OUT_DIR / 'checkpoints').exists() else None
print(f'이어서 학습: {ckpt}' if ckpt else '처음부터 학습')
stats = trainer.train(resume_from_checkpoint=ckpt)
print(f"\n끝: {stats.metrics['train_runtime'] / 60:.1f}분, 최종 loss {stats.metrics['train_loss']:.4f}, "
      f"최대 VRAM {torch.cuda.max_memory_reserved() / 2**30:.1f}GB")

### 7-1. 학습 곡선
왼쪽: train loss(학습, 선)·eval loss(평가, 점). Colab에 한글 글꼴이 없어 그래프 글자는 영어. 오른쪽: 학습률이 스텝마다 어떻게 바뀌었는지.

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
series = lambda key: [(h['step'], h[key]) for h in hist if key in h]
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8))
if series('loss'):
    a.plot(*zip(*series('loss')), label='train loss')
if series('eval_loss'):
    a.plot(*zip(*series('eval_loss')), 'o-', label='eval loss')
a.set_xlabel('step'); a.set_title('loss'); a.legend(); a.grid(alpha=.3)
if series('learning_rate'):
    b.plot(*zip(*series('learning_rate')), color='tab:orange')
b.set_xlabel('step'); b.set_title('learning rate'); b.grid(alpha=.3)
plt.tight_layout(); plt.show()
for h in hist:
    if 'eval_loss' in h:
        print(f"step {h['step']:>4}: 평가 loss {h['eval_loss']:.4f}")

## 8. 평가: 첫 도구를 맞게 부르나
`eval.jsonl`(학습에 안 쓴 원본)에서 마지막 질문까지만 주고, 모델이 처음 부른 도구가 정답 대화의 첫 도구와 같은지 센다.
정답이 도구 없이 바로 답하는 턴(확인 질문 등)이면 도구를 안 불러야 정답.

In [ ]:
TOOL_CALL = re.compile(r'<tool_call>\s*<function=([\w-]+)>(.*?)</function>\s*</tool_call>', re.S)
PARAM = re.compile(r'<parameter=([\w-]+)>\n?(.*?)\n?</parameter>', re.S)

def parse_calls(text):
    """Qwen3.5 도구 호출 → [(이름, 인자)]. 값이 JSON이면 풀어서."""
    calls = []
    for name, body in TOOL_CALL.findall(text):
        args = {}
        for k, v in PARAM.findall(body):
            try:
                args[k] = json.loads(v)
            except json.JSONDecodeError:
                args[k] = v
        calls.append((name, args))
    return calls

def generate(msgs, max_new_tokens=1024):
    ids = tok(render(msgs, add_generation_prompt=True), return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.8, do_sample=True)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)

model.eval()
hit, total = Counter(), Counter()
def split_turn(msgs):
    """마지막 사용자 질문까지(입력)와 그다음 첫 도구 이름(정답, 도구 없이 답하면 None)."""
    last = max(i for i, m in enumerate(msgs) if m['role'] == 'user')
    nxt = msgs[last + 1]
    return msgs[:last + 1], (nxt['tool_calls'][0]['function']['name'] if nxt.get('tool_calls') else None)

for r in eval_rows[:40]:
    prompt_msgs, gold = split_turn(r['messages'])
    text = generate(prompt_msgs, max_new_tokens=400)
    calls = parse_calls(text)
    total[r['kind']] += 1
    hit[r['kind']] += (calls[0][0] == gold) if calls else gold is None
for k in total:
    print(f'{k}: 첫 도구 정답 {hit[k]}/{total[k]}')
print(f'전체 {sum(hit.values())}/{sum(total.values())}')
print('\n마지막 예시 출력:\n', text[:600])

## 9. 저장
LoRA 어댑터만 저장한다 (수십 MB). 서버에서는 원본 모델 + 어댑터로 불러 쓴다.

In [ ]:
ADAPTER = OUT_DIR / 'qwen35-4b-coach-lora'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print('저장:', ADAPTER)

In [ ]:
# 드라이브를 안 쓰면 런타임이 끊길 때 사라지므로 내려받기 (드라이브면 pokemon_coach/outputs에 이미 있음)
if not USE_DRIVE:
    from google.colab import files
    shutil.make_archive(str(ADAPTER), 'zip', ADAPTER)
    files.download(str(ADAPTER) + '.zip')

## 10. 채팅방
학습한 모델과 자유롭게 대화한다. 서버(`llm/agent/loop.py`)와 같은 방식으로 **실제 도구를 실행**한다
(back과 DB를 이 프로세스 안에서 부름). 답변 아래에 부른 도구와 추천 파티가 같이 보인다.
- 포맷(싱글/더블)은 채팅창 아래에서 바꾼다. 모델이 `propose_party`로 띄운 파티는 다음 질문의 "현재 파티"가 된다
- 대화를 지우면(🗑) 파티도 비워진다

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import gradio as gr
from inproc_back import back_client          # back(Django) + DB를 이 프로세스에서 (서버 안 띄움)
from agent.tools import Tools
from apps.api.common import names
from apps.dex.models import Ruleset

MAX_ROUNDS = 6                                # 한 번 답할 때 도구를 주고받는 최대 횟수 (서버와 같게)
RID = Ruleset.objects.order_by('-start_date').first().id
CHAT = {'turns': [], 'party': []}            # 지난 대화(사용자, 최종 답변) · 현재 파티

def ko(kind, key):
    row = names(RID)[kind].get(key) or {}
    return row.get('name_ko') or key

def readable(s):
    sp = ' '.join(f'{k}{v}' for k, v in (s.get('sp') or {}).items() if v)
    return (f"{ko('pokemon', s['pokemon'])} @ {ko('item', s.get('item', '')) or '-'} / {ko('ability', s.get('ability', ''))} / "
            f"{ko('nature', s.get('nature', ''))} / SP {sp} / " + ', '.join(ko('move', m) for m in s.get('moves', []) if m))

def clean(text):
    for pat in (r'<think>.*?</think>', r'<tool_call>.*?</tool_call>', r'<\|im_end\|>.*', r'<\|endoftext\|>'):
        text = re.sub(pat, '', text, flags=re.S)
    return text.strip()

async def agent_turn(question, fmt):
    """서버의 build_messages + run_agent와 같은 흐름: 지난 대화는 글만, 마지막 질문 앞에 [화면] 상황."""
    msgs = [{'role': 'system', 'content': SYSTEM}]
    for q, a in CHAT['turns']:
        msgs += [{'role': 'user', 'content': q}, {'role': 'assistant', 'content': a}]
    party = CHAT['party'] + [None] * (6 - len(CHAT['party']))
    msgs.append({'role': 'user', 'content': screen_context(fmt, party) + '\n\n' + question})
    log = []
    async with back_client() as c:
        tools = Tools(c, fmt)
        for _ in range(MAX_ROUNDS):
            text = generate(msgs)
            calls, reply = parse_calls(text), clean(text)
            if not calls:
                return reply, log
            msgs.append({'role': 'assistant', 'content': reply, 'tool_calls': [
                {'type': 'function', 'function': {'name': n, 'arguments': a}} for n, a in calls]})
            for n, a in calls:
                try:
                    out = json.dumps(await tools.run(n, a), ensure_ascii=False)
                except Exception as e:                 # 없는 ID, 잘못된 입력 → 모델에게 오류로 돌려줌 (서버와 같게)
                    out = f'오류: {e}'
                msgs.append({'role': 'tool', 'content': out})
                log.append(f'🔧 {n}({json.dumps(a, ensure_ascii=False)[:120]}) → {out[:100]}')
                if n == 'propose_party' and tools.proposed:
                    CHAT['party'] = tools.proposed
                    tools.proposed = None
                    log.append('📋 추천 파티를 현재 파티로:\n' + '\n'.join(f'  {i}. {readable(s)}' for i, s in enumerate(CHAT['party'], 1)))
    return '(도구 호출이 너무 많아 멈췄어요)', log

def respond(message, history, fmt):
    if not history:                              # 새 대화 (지우기 포함)
        CHAT['turns'], CHAT['party'] = [], []
    fmt = 'doubles' if fmt == '더블' else 'singles'
    t0 = time.time()
    reply, log = ThreadPoolExecutor(1).submit(asyncio.run, agent_turn(message, fmt)).result()
    CHAT['turns'].append((message, reply))
    tail = f'\n\n---\n' + '\n'.join(log) if log else ''
    return f'{reply}{tail}\n\n_({time.time() - t0:.0f}초)_'

fmt_box = gr.Radio(['더블', '싱글'], value='더블', label='포맷')
try:
    demo = gr.ChatInterface(respond, additional_inputs=[fmt_box], type='messages', title='파티 코치 (파인튜닝 모델)',
                            examples=[['한카리아스 더블 샘플 짜줘', '더블'], ['요즘 싱글 픽률 1위 뭐야?', '싱글']])
except TypeError:                                # type 인자가 없는 gradio 버전
    demo = gr.ChatInterface(respond, additional_inputs=[fmt_box], title='파티 코치 (파인튜닝 모델)')
demo.launch(debug=False)

### (선택) 병합·GGUF
llama.cpp/Ollama로 돌리려면 4bit GGUF로 내보낸다 (추론 약 3GB). 시간이 오래 걸리니 필요할 때만.
```python
model.save_pretrained_gguf(str(OUT_DIR / 'gguf'), tokenizer, quantization_method='q4_k_m')
```